# 03. Spatial Transcriptomics, Isoform DTU, Epigenomics & Quantitative Proteomics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gillanilab/pythonic_biotools/blob/main/notebooks/03_spatial_singlecell_epigenomics_and_proteomics.ipynb)

This multi-omics tutorial connects spatial transcriptomics, alternative splicing, DNA methylation, chromatin accessibility, transcription factor motif selection, and TMT/DIA quantitative proteomics using six packages from `pythonic_biotools`:

1. **`cside`** (`spacexr`) — Cell-type-specific differential expression in spatial transcriptomics on `AnnData`.
2. **`saturn`** (`satuRn` & `stageR`) — Quasi-binomial Empirical Bayes differential transcript usage (`DTU`) and two-stage OFDR control.
3. **`dss`** — Beta-Binomial Empirical Bayes bisulfite / EM-seq differential methylation (`DML` & `DMR`).
4. **`csaw`** — Peak-free sliding-window differential ChIP-seq / ATAC-seq binding with Simes region-level FDR control.
5. **`monalisa`** — GC/dinucleotide-matched binned motif enrichment and Randomized Lasso stability selection.
6. **`deqms`** — Peptide/PSM-count-dependent Empirical Bayes protein differential abundance.

In [1]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "anndata",
            "git+https://github.com/gillanilab/pythonic_biotools.git",
        ]
    )
else:
    repo_root = Path.cwd().resolve()
    if not (repo_root / "pythonic_biotools").is_dir() and (repo_root.parent / "pythonic_biotools").is_dir():
        repo_root = repo_root.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

import anndata as ad
import numpy as np
import pandas as pd

from pythonic_biotools import csaw, cside, deqms, dss, monalisa, saturn

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 15)
print("Loaded multi-omics toolkit: cside, saturn, dss, csaw, monalisa, deqms")

Loaded multi-omics toolkit: cside, saturn, dss, csaw, monalisa, deqms


## 1. Spatial Cell-Type-Specific Differential Expression on `AnnData` (`cside`)

In spatial Visium/Slide-seq spots containing mixtures of `Tumor` and `CAF_Stroma` cells, `run_cside_single` disentangles which specific cell type upregulates hypoxia response genes (`VEGFA`, `CA9`) along a spatial microenvironment gradient.

In [2]:
rng = np.random.default_rng(303)
n_spots = 120
hypoxia = np.linspace(0.0, 1.0, n_spots)
w_tumor = rng.uniform(0.25, 0.85, size=n_spots)
weights = np.column_stack([w_tumor, 1.0 - w_tumor])
n_umi = np.full(n_spots, 2500.0)

gene_names = ["VEGFA", "CA9", "COL1A1", "ACTA2"] + [f"HK_{i}" for i in range(10)]
rates = np.zeros((n_spots, len(gene_names)))
for g_idx, g_name in enumerate(gene_names):
    tumor_lfc = 1.6 if g_name in ("VEGFA", "CA9") else 0.0
    rates[:, g_idx] = n_umi * (weights[:, 0] * 0.018 * np.exp(tumor_lfc * hypoxia) + weights[:, 1] * 0.018)

adata_sp = ad.AnnData(X=rng.poisson(rates).astype(float))
adata_sp.var_names = gene_names
adata_sp.obsm["cell_type_weights"] = pd.DataFrame(weights, index=adata_sp.obs_names, columns=["Tumor", "CAF_Stroma"])
adata_sp.obs["hypoxia"] = hypoxia

cside_res = cside.run_cside_single(adata_sp, explanatory_variable=adata_sp.obs["hypoxia"], settings="default")
cside_df = cside_res.to_dataframe()
print("Top spatial cell-type-specific DE hits (CSIDE):")
print(cside_df.sort_values("p_val").head(6).round(4).to_string(index=False))

Top spatial cell-type-specific DE hits (CSIDE):
cell_type   gene  log_fc     se  z_stat  p_val  q_val  significant  converged  sigma_gene
    Tumor  VEGFA  1.1782 0.0842 13.9863 0.0000 0.0000         True       True      0.0109
    Tumor    CA9  1.0723 0.0866 12.3764 0.0000 0.0000         True       True      0.0109
    Tumor   HK_7 -0.6100 0.1584 -3.8499 0.0001 0.0005         True       True      0.0348
    Tumor   HK_3 -0.6186 0.1618 -3.8228 0.0001 0.0005         True       True      0.0109
    Tumor COL1A1 -0.5457 0.1503 -3.6299 0.0003 0.0008         True       True      0.0109
    Tumor   HK_2 -0.5054 0.1474 -3.4278 0.0006 0.0014         True       True      0.0109


## 2. Isoform Differential Transcript Usage & Stage-Wise OFDR (`saturn`)

Next, we test isoform switching (`PKM_txA`/`PKM_txB` and `CD44_txA`/`CD44_txB`) across 12 samples (`6 Normoxia` vs. `6 Hypoxia`) using quasi-binomial Empirical Bayes dispersion moderation and `stageR` two-stage gene/transcript FDR screening.

In [3]:
n_samp = 12
cond = np.array([0] * 6 + [1] * 6)
tx_rows, tx2gene = [], []
gene_defs = [("PKM", True), ("CD44", True)] + [(f"GENE_{k}", False) for k in range(28)]

for g_name, is_switched in gene_defs:
    prop_a = np.where(cond == 1, 0.78, 0.22) if is_switched else np.full(n_samp, 0.50)
    tot_cnt = rng.integers(90, 160, size=n_samp)
    cnt_a = rng.binomial(tot_cnt, prop_a)
    tx_rows.extend([cnt_a, tot_cnt - cnt_a])
    tx2gene.extend(
        [
            {"transcript_id": f"{g_name}_txA", "gene_id": g_name},
            {"transcript_id": f"{g_name}_txB", "gene_id": g_name},
        ]
    )

tx_df = pd.DataFrame(
    np.vstack(tx_rows),
    index=[r["transcript_id"] for r in tx2gene],
    columns=[f"Sample_{i:02d}" for i in range(n_samp)],
)
design_dtu = pd.DataFrame({"intercept": 1.0, "hypoxia": cond}, index=tx_df.columns)

filt_dtu = saturn.filter_dtu(tx_df, pd.DataFrame(tx2gene), min_count=5, min_samples=3)
dtu_fit = saturn.fit_dtu(filt_dtu.counts, filt_dtu.tx2gene, design_dtu, settings="default")
dtu_tab = saturn.test_dtu(dtu_fit, contrast=[0.0, 1.0], sort=True, settings="default")
stage_tab = saturn.stagewise_dtu(dtu_tab, alpha=0.05, pval_col="pval_regular")

print("Top differential transcript usage isoforms (satuRn + stageR):")
print(stage_tab.head(6).round(5).to_string())

Top differential transcript usage isoforms (satuRn + stageR):
              transcript_id  gene_id  pval_transcript  pval_gene  padj_gene  padj_transcript  gene_significant  tx_significant
transcript_id                                                                                                                 
PKM_txB             PKM_txB      PKM          0.00000    0.00000    0.00000              0.0              True            True
PKM_txA             PKM_txA      PKM          0.00000    0.00000    0.00000              0.0              True            True
CD44_txB           CD44_txB     CD44          0.00000    0.00000    0.00000              0.0              True            True
CD44_txA           CD44_txA     CD44          0.00000    0.00000    0.00000              0.0              True            True
GENE_18_txB     GENE_18_txB  GENE_18          0.12291    0.12291    0.93138              1.0             False           False
GENE_18_txA     GENE_18_txA  GENE_18          0.1

## 3. Bisulfite Differential Methylation (`dss`) & Peak-Free ChIP/ATAC Differential Binding (`csaw`)

We jointly profile an enhancer locus across 3 Control vs. 3 Treated replicates using `dss` (`dml_test` + `call_dmr`) for CpG methylation and `csaw` (`window_counts` + `glm_ql_fit_test` + `merge_windows` + `combine_tests`) for sliding-window chromatin accessibility.

In [4]:
# 3A. DSS bisulfite CpG methylation
cpg_pos = np.arange(1000, 1000 + 60 * 35, 35)
ctrl_bs, trt_bs = [], []
for _ in range(3):
    cov_n = rng.integers(28, 48, size=len(cpg_pos))
    p_trt = np.where((cpg_pos >= 1450) & (cpg_pos <= 1900), 0.82, 0.18)
    ctrl_bs.append(pd.DataFrame({"chr": "chr1", "pos": cpg_pos, "N": cov_n, "X": rng.binomial(cov_n, 0.18)}))
    trt_bs.append(pd.DataFrame({"chr": "chr1", "pos": cpg_pos, "N": cov_n, "X": rng.binomial(cov_n, p_trt)}))

dml_df = dss.dml_test(ctrl_bs, trt_bs, settings="default")
dmr_df = dss.call_dmr(dml_df, p_threshold=1e-3, minCG=4, minlen=80, settings="default")
print("DSS Differentially Methylated Regions (DMRs):")
print(dmr_df.round(4).to_string(index=False))

# 3B. csaw sliding-window differential ATAC/ChIP accessibility
frag_libs = []
for s_idx in range(6):
    n_enhancer = 110 if s_idx >= 3 else 18
    pos_reads = np.concatenate([rng.integers(1500, 1750, size=n_enhancer), rng.integers(3400, 3650, size=40)])
    frag_libs.append(pd.DataFrame({"chrom": "chr1", "pos": pos_reads, "strand": "+", "mapq": 42}))

wdata = csaw.window_counts(fragments=frag_libs, chrom_sizes={"chr1": 5000}, settings="default")
ql_res = csaw.glm_ql_fit_test(wdata, design=np.column_stack([np.ones(6), [0, 0, 0, 1, 1, 1]]), settings="default")
merged = csaw.merge_windows(wdata, tol=120, settings="default")
regions_csaw = csaw.combine_tests(merged, ql_res, settings="default")
print("\ncsaw Merged Differential Chromatin Regions:")
print(regions_csaw.sort_values("FDR").head(4).round(5).to_string(index=False))

DSS Differentially Methylated Regions (DMRs):
 chr  start  end  length  nCG  meanMethy1  meanMethy2  diff.Methy  areaStat
chr1   1350 1945     595   18       0.176      0.6335     -0.4575 -359.1267

csaw Merged Differential Chromatin Regions:
 region_id chrom  start  end  width  num_tests  num_up_logFC  num_down_logFC  rep_test  rep_logFC direction  pval  FDR
         0  chr1   1400 1950    550          9             9               0         4    1.22742        up   0.0  0.0
         1  chr1   3300 3850    550          9             0               9        13   -1.37084      down   0.0  0.0


## 4. Sequence-Composition-Matched Motif Enrichment & Lasso Stability Selection (`monalisa`)

We test which transcription factor motif explains enhancer remodeling across binned regions (`hyper` vs. `neutral`), comparing `settings="default"` (`pseudocount=1.0`) with `settings="R_default"` (`pseudocount.log2enr=8.0`), followed by Complementary-Pairs Randomized Lasso stability selection.

In [5]:
pfm_hif = np.array([[5, 92, 4, 5, 90, 4], [4, 3, 90, 4, 4, 90], [88, 3, 4, 88, 3, 4], [3, 2, 2, 3, 3, 2]], float)
pwms = {"HIF1A_RCGTG": monalisa.pfm_to_pwm(pfm_hif)}

enhancer_seqs = ["ATGCACGTGCATGCACGTGC" * 4] * 35 + ["ATGCTAGCTAGCATGCTAGC" * 4] * 35
delta_meth = np.concatenate([rng.normal(0.58, 0.08, 35), rng.normal(0.00, 0.08, 35)])
bins_ml = np.where(delta_meth > 0.25, "hyper", "neutral")

enr_def = monalisa.calc_binned_motif_enrichment(enhancer_seqs, bins_ml, pwms=pwms, settings="default")
enr_r = monalisa.calc_binned_motif_enrichment(enhancer_seqs, bins_ml, pwms=pwms, settings="R_default")

print("monaLisa binned enrichment (settings='default'):")
print(enr_def.to_dataframe()[["motif", "bin", "log2enr", "neg_log10_padj"]].round(3).to_string(index=False))
print("\nmonaLisa binned enrichment (settings='R_default', pseudocount=8.0):")
print(enr_r.to_dataframe()[["motif", "bin", "log2enr", "neg_log10_padj"]].round(3).to_string(index=False))

motif_scan = monalisa.scan_sequences_pwm(enhancer_seqs, pwms)
X_motifs = pd.DataFrame(
    np.column_stack([motif_scan.counts, rng.poisson(1.0, size=(70, 4))]),
    columns=["HIF1A_RCGTG", "DECOY_TF1", "DECOY_TF2", "DECOY_TF3", "DECOY_TF4"],
)
stab = monalisa.rand_lasso_stab_sel(X_motifs, delta_meth, n_subsamples=60, cutoff=0.7, seed=42, settings="default")
print("\nRandomized Lasso stability selection probabilities:")
print(stab.sel_prob.round(3).to_string())

monaLisa binned enrichment (settings='default'):
      motif     bin  log2enr  neg_log10_padj
HIF1A_RCGTG   hyper      0.0             0.0
HIF1A_RCGTG neutral      0.0             0.0

monaLisa binned enrichment (settings='R_default', pseudocount=8.0):
      motif     bin  log2enr  neg_log10_padj
HIF1A_RCGTG   hyper      0.0             0.0
HIF1A_RCGTG neutral      0.0             0.0

Randomized Lasso stability selection probabilities:
feature
HIF1A_RCGTG    0.000
DECOY_TF1      0.550
DECOY_TF2      0.258
DECOY_TF3      0.400
DECOY_TF4      0.200


## 5. PSM-Count-Moderated TMT Quantitative Proteomics (`deqms`)

In isobaric TMT or label-free proteomics, proteins quantified by 12+ PSMs exhibit much lower technical variance than single-peptide proteins. `deqms` fits a prior variance trend as a function of $\log_2(\text{PSM count})$.

In [6]:
psm_rows = []
for p_idx in range(45):
    prot_name = "CA9_PROT" if p_idx == 0 else ("VEGFA_PROT" if p_idx == 1 else f"BG_PROT_{p_idx:02d}")
    n_psms = 14 if p_idx < 12 else 2
    lfc_true = 1.45 if p_idx == 0 else (1.10 if p_idx == 1 else 0.0)
    for _ in range(n_psms):
        intensities = np.concatenate([rng.normal(20.0, 0.24, 3), rng.normal(20.0 + lfc_true, 0.24, 3)])
        row = {"protein": prot_name}
        row.update({f"TMT_{k + 1}": intensities[k] for k in range(6)})
        psm_rows.append(row)

swept = deqms.median_sweeping(pd.DataFrame(psm_rows), group_col="protein", settings="default")
design_tmt = np.column_stack([np.ones(6), [0, 0, 0, 1, 1, 1]])
deq_fit = deqms.spectra_count_ebayes(swept.abundance, design=design_tmt, count=swept.count, settings="R_default")
deq_table = deqms.output_result(deq_fit, coef=1, settings="R_default")

print("Top differentially abundant proteins (DEqMS spectraCounteBayes):")
print(deq_table[["logFC", "AveExpr", "count", "sca.t", "sca.P.Value", "sca.adj.pval"]].head(5).round(5).to_string())

Top differentially abundant proteins (DEqMS spectraCounteBayes):
              logFC  AveExpr  count     sca.t  sca.P.Value  sca.adj.pval
gene                                                                    
CA9_PROT    1.41826  0.05530     14  25.60273      0.00000       0.00000
VEGFA_PROT  1.17409 -0.00649     14  21.19490      0.00000       0.00000
BG_PROT_26  0.34793 -0.01954      2   2.34097      0.01923       0.26865
BG_PROT_42 -0.33575 -0.01728      2  -2.25906      0.02388       0.26865
BG_PROT_18 -0.30437 -0.01157      2  -2.04793      0.04057       0.36511
